# Pertemuan 6 - Persiapan Data (Data Preprocessing)
**Nama:** RANU RATMAJA
**NIM:** 230401010104
**Prodi:** Sistem Informasi
**Universitas:** Universitas Siber Asia (UNSIA)
**Tahun:** 2023

## Tujuan Praktikum
Pada pertemuan ini mahasiswa mempelajari teknik **Persiapan Data (Data Preprocessing)** sebelum data digunakan untuk pemodelan machine learning. Materi mencakup: normalisasi/standarisasi, encoding variabel kategorikal, feature selection, dan pembagian data training-testing.

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.datasets import load_iris
print('Library berhasil diimport!')

## 2. Load Dataset & Tambahkan Kolom Kategorikal

In [2]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df['lokasi'] = np.random.choice(['Lab A', 'Lab B', 'Lab C'], size=len(df))
df['kondisi'] = np.random.choice(['Baik', 'Sedang', 'Buruk'], size=len(df), p=[0.6,0.3,0.1])
print('Dataset dengan kolom tambahan:')
print(df.head(8))
print('\nShape:', df.shape)

## 3. Normalisasi - Min-Max Scaler

In [3]:
num_cols = iris.feature_names
minmax = MinMaxScaler()
df_minmax = pd.DataFrame(minmax.fit_transform(df[num_cols]), columns=num_cols)
print('=== SEBELUM Normalisasi ===')
print(df[num_cols].describe().round(3))
print('\n=== SETELAH Min-Max Normalisasi (range 0-1) ===')
print(df_minmax.describe().round(3))

## 4. Standarisasi - Standard Scaler (Z-Score)

In [4]:
std_scaler = StandardScaler()
df_std = pd.DataFrame(std_scaler.fit_transform(df[num_cols]), columns=num_cols)
print('=== SETELAH Standarisasi Z-Score ===')
print(df_std.describe().round(3))

# Visualisasi perbandingan
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].boxplot(df['sepal length (cm)'])
axes[0].set_title('Original Data')
axes[1].boxplot(df_minmax['sepal length (cm)'])
axes[1].set_title('Min-Max Normalized')
axes[2].boxplot(df_std['sepal length (cm)'])
axes[2].set_title('Z-Score Standardized')
plt.suptitle('Perbandingan Scaling: Sepal Length', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Encoding Variabel Kategorikal

In [5]:
df_enc = df.copy()
# Label Encoding untuk 'kondisi' (ordinal)
le = LabelEncoder()
df_enc['kondisi_encoded'] = le.fit_transform(df_enc['kondisi'])
print('Label Encoding - kondisi:')
print(dict(zip(le.classes_, le.transform(le.classes_))))

# One-Hot Encoding untuk 'lokasi' (nominal)
lokasi_dummies = pd.get_dummies(df_enc['lokasi'], prefix='lokasi')
df_enc = pd.concat([df_enc, lokasi_dummies], axis=1)
print('\nOne-Hot Encoding - lokasi:')
print(df_enc[['lokasi', 'lokasi_Lab A', 'lokasi_Lab B', 'lokasi_Lab C']].head(8))

## 6. Feature Selection

In [6]:
X = df[num_cols]
y = iris.target
selector = SelectKBest(score_func=f_classif, k=2)
selector.fit(X, y)
scores = pd.Series(selector.scores_, index=num_cols)
print('F-Score setiap fitur:')
print(scores.sort_values(ascending=False).round(3))
selected = scores.sort_values(ascending=False).index[:2].tolist()
print(f'\n2 Fitur terbaik: {selected}')
plt.figure(figsize=(8, 4))
scores.sort_values(ascending=False).plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Feature Importance (ANOVA F-Score)')
plt.ylabel('F-Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 7. Split Data Training & Testing

In [7]:
X_scaled = pd.DataFrame(std_scaler.fit_transform(X), columns=num_cols)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print('=== HASIL SPLIT DATA ===')
print(f'Total data    : {len(X_scaled)} sampel')
print(f'Training set  : {X_train.shape[0]} sampel ({X_train.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'Testing set   : {X_test.shape[0]} sampel ({X_test.shape[0]/len(X_scaled)*100:.0f}%)')
print(f'\nDistribusi kelas training: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Distribusi kelas testing : {dict(zip(*np.unique(y_test, return_counts=True)))}')
print('\nData siap digunakan untuk pemodelan Machine Learning!')

## Kesimpulan
- **Normalisasi Min-Max** mengubah data ke rentang [0,1]; cocok jika tidak ada asumsi distribusi normal.
- **Standarisasi Z-Score** mengubah data agar mean=0 dan std=1; cocok untuk algoritma berbasis jarak.
- **Label Encoding** digunakan untuk variabel kategorikal ordinal (ada urutan).
- **One-Hot Encoding** digunakan untuk variabel kategorikal nominal (tidak ada urutan).
- **Feature Selection** memilih fitur yang paling relevan untuk mengurangi dimensi dan noise.
- **Train-Test Split** dengan stratify memastikan proporsi kelas seimbang di data training dan testing.